# Ensembles and multiple testing

Two problems show up the moment you start scoring more than one column:

1. **Detector disagreement.** Each detector catches a different *kind* of shift. Voting across
   several reduces both false positives (single-detector overfit) and false negatives
   (single-detector blind spots).
2. **Multiple testing.** If you score 50 columns at p<0.05, you expect ~2.5 false alarms
   per run *under the null*. The Benjamini–Hochberg procedure bounds the false discovery rate
   so the alert load stays sane.

In [ ]:
import numpy as np
import pandas as pd

from drift_control import EnsembleDriftDetector, UnifiedDriftDetector
from drift_control.multiple_testing import adjust_pvalues

rng = np.random.default_rng(0)

## 1. Voting across detectors

`EnsembleDriftDetector` runs each method per column and decides via majority/any/all vote.
Here we build a 4-detector ensemble and run it on a 6-column frame where only 2 columns
have actually drifted.

In [ ]:
n = 400
df_prior = pd.DataFrame({
    "x1": rng.normal(0, 1, n),
    "x2": rng.normal(0, 1, n),
    "x3": rng.normal(0, 1, n),
    "x4": rng.normal(0, 1, n),
    "x5": rng.normal(0, 1, n),
    "x6": rng.normal(0, 1, n),
})
df_post = pd.DataFrame({
    "x1": rng.normal(0.8, 1, n),    # drifted (mean shift)
    "x2": rng.normal(0, 1.8, n),    # drifted (variance shift)
    "x3": rng.normal(0, 1, n),      # clean
    "x4": rng.normal(0, 1, n),      # clean
    "x5": rng.normal(0, 1, n),      # clean
    "x6": rng.normal(0, 1, n),      # clean
})

ensemble = EnsembleDriftDetector(
    methods=["psi", "ks", "cvm", "js"],
    vote_mode="majority",
)
results = ensemble.detect_drift(df_prior, df_post)

for col, res in results.items():
    print(
        f"  {col}: drift={res.drift_detected}  votes={res.votes}/{res.required_votes}"
    )

Notice the ensemble correctly flags `x1` and `x2` and leaves the clean columns alone,
even though any *single* detector might have fired on one of the clean columns by chance.

## 2. Multiple testing across many columns

Now imagine 50 columns, all drawn from the null. A naive p<0.05 cutoff gives roughly 2–3
spurious flags. Adjusting the p-values via Benjamini–Hochberg bounds the *expected proportion*
of false discoveries instead, which is usually what you want when alerting humans.

In [ ]:
alpha = 0.05
raw_pvalues: list[float] = []
ks = UnifiedDriftDetector(method="ks")
for col in range(50):
    ref = rng.normal(0, 1, size=300)
    cur = rng.normal(0, 1, size=300)  # same distribution -> all null
    out = ks.detect_drift(ref, cur)
    raw_pvalues.append(out.p_value)

raw_alarms = sum(p < alpha for p in raw_pvalues)
bh_alarms = sum(p < alpha for p in adjust_pvalues(raw_pvalues, method="bh"))
bonf_alarms = sum(p < alpha for p in adjust_pvalues(raw_pvalues, method="bonferroni"))

print(f"raw p<{alpha}        : {raw_alarms} alarms out of 50 null columns")
print(f"after BH correction  : {bh_alarms} alarms")
print(f"after Bonferroni     : {bonf_alarms} alarms")

## 3. Mixed case: some drifted, some not

Here 5 columns out of 50 actually drift. BH should retain most of the real positives while
killing nearly all of the false ones.

In [ ]:
drifted_cols = set(range(5))
pvalues = []
for col in range(50):
    ref = rng.normal(0, 1, size=300)
    if col in drifted_cols:
        cur = rng.normal(0.7, 1, size=300)  # real shift
    else:
        cur = rng.normal(0, 1, size=300)    # null
    pvalues.append(ks.detect_drift(ref, cur).p_value)

adjusted = adjust_pvalues(pvalues, method="bh")
true_pos = sum(1 for i, p in enumerate(adjusted) if i in drifted_cols and p < alpha)
false_pos = sum(1 for i, p in enumerate(adjusted) if i not in drifted_cols and p < alpha)
print(f"true positives (5 expected):  {true_pos}")
print(f"false positives (~0 expected): {false_pos}")

## When to use which

- **Bonferroni**: simple, very conservative. Good when each alarm is expensive to investigate
  and you'd rather miss real shifts than chase false ones.
- **BH (FDR)**: bounds the fraction of alarms that are false; usually the right default when you
  expect at least *some* real drift.
- **No correction**: only if you're scoring a single column or doing exploratory analysis.

The CLI exposes this directly: `drift-control ... --correction bh`.